In [1]:
import pandas as pd
df = pd.read_csv('../data/raw/loan.csv', low_memory=False, nrows=5)
df.shape

(5, 145)

In [1]:
import pandas as pd

cols_needed = ['loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'sub_grade',
               'emp_length', 'home_ownership', 'annual_inc', 'purpose', 'dti',
               'open_acc', 'revol_bal', 'revol_util', 'total_acc', 'loan_status']

chunks = []
for chunk in pd.read_csv('../data/raw/loan.csv', low_memory=False, usecols=cols_needed, chunksize=100000):
    chunk = chunk[chunk['loan_status'].isin(['Fully Paid', 'Charged Off'])]
    chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)
df.shape

(1303607, 16)

In [2]:
df.info()
df['loan_status'].value_counts()
df.isnull().mean().sort_values(ascending=False)

<class 'pandas.DataFrame'>
RangeIndex: 1303607 entries, 0 to 1303606
Data columns (total 16 columns):
 #   Column          Non-Null Count    Dtype  
---  ------          --------------    -----  
 0   loan_amnt       1303607 non-null  int64  
 1   term            1303607 non-null  str    
 2   int_rate        1303607 non-null  float64
 3   installment     1303607 non-null  float64
 4   grade           1303607 non-null  str    
 5   sub_grade       1303607 non-null  str    
 6   emp_length      1228153 non-null  str    
 7   home_ownership  1303607 non-null  str    
 8   annual_inc      1303607 non-null  float64
 9   loan_status     1303607 non-null  str    
 10  purpose         1303607 non-null  str    
 11  dti             1303295 non-null  float64
 12  open_acc        1303607 non-null  float64
 13  revol_bal       1303607 non-null  int64  
 14  revol_util      1302797 non-null  float64
 15  total_acc       1303607 non-null  float64
dtypes: float64(7), int64(2), str(7)
memory usage: 1

emp_length        0.057881
revol_util        0.000621
dti               0.000239
loan_amnt         0.000000
installment       0.000000
grade             0.000000
int_rate          0.000000
term              0.000000
home_ownership    0.000000
sub_grade         0.000000
loan_status       0.000000
annual_inc        0.000000
purpose           0.000000
open_acc          0.000000
revol_bal         0.000000
total_acc         0.000000
dtype: float64

In [3]:
  df['loan_status'].value_counts()

loan_status
Fully Paid     1041952
Charged Off     261655
Name: count, dtype: int64

In [4]:
# ## Data Exploration Notes

# **Dataset shape:** 1,303,607 rows × 16 columns (after filtering to loans with a known final outcome)

# **What `loan_status` values mean:**
# - `Fully Paid` (1,041,952 loans, ~80%) — the borrower repaid the loan in full. Counts as **not default** (target = 0).
# - `Charged Off` (261,655 loans, ~20%) — the borrower defaulted and the lender wrote off the loan as a loss. Counts as **default** (target = 1).
# - Other original values like `Current`, `Late`, `In Grace Period` were excluded, since those loans haven't reached a final outcome yet — including them would leak information the model shouldn't have access to.

# **Class balance:** ~80% paid vs ~20% defaulted — meaningfully imbalanced, so `class_weight='balanced'` will be used during modeling.

# **Columns >50% empty, or IDs/URLs/free text:**
# - None. All 16 columns selected during loading were chosen specifically as clean, application-time features, so there was nothing to drop for missingness or irrelevance. (Worst missingness: `emp_length` at ~5.8%.)

In [7]:
df = df[df['loan_status'].isin(['Fully Paid', 'Charged Off'])].copy()
df['default'] = (df['loan_status'] == 'Charged Off').astype(int)

In [8]:
leak_cols = ['recoveries', 'total_pymnt', 'total_pymnt_inv', 'last_pymnt_d',
             'last_pymnt_amnt', 'next_pymnt_d', 'total_rec_prncp',
             'total_rec_int', 'total_rec_late_fee', 'collection_recovery_fee']
df = df.drop(columns=[c for c in leak_cols if c in df.columns])

In [9]:
for col in df.select_dtypes(include='number').columns:
    df[col] = df[col].fillna(df[col].median())
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].fillna('Unknown')

C:\Users\91988\AppData\Local\Temp\ipykernel_30284\2217783964.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include='object').columns:


In [11]:
df.isnull().sum().sum()


np.int64(0)

In [12]:
X = df.drop(columns=['loan_status', 'default'])
y = df['default']

X = pd.get_dummies(X, columns=['home_ownership', 'purpose'], drop_first=True)

grade_map = {'A':1,'B':2,'C':3,'D':4,'E':5,'F':6,'G':7}
X['grade'] = X['grade'].map(grade_map)
X = X.drop(columns=['sub_grade'])

X['emp_length'] = X['emp_length'].replace({
    '< 1 year':0,'1 year':1,'2 years':2,'3 years':3,'4 years':4,
    '5 years':5,'6 years':6,'7 years':7,'8 years':8,'9 years':9,'10+ years':10
})
X['emp_length'] = pd.to_numeric(X['emp_length'], errors='coerce').fillna(0)

X['term'] = X['term'].str.extract(r'(\d+)').astype(int)

X.head()

,loan_amnt,term,int_rate,installment,grade,emp_length,annual_inc,dti,open_acc,revol_bal,...,purpose_home_improvement,purpose_house,purpose_major_purchase,purpose_medical,purpose_moving,purpose_other,purpose_renewable_energy,purpose_small_business,purpose_vacation,purpose_wedding
0,30000,36,22.35,1151.16,4,5.0,100000.0,30.46,11.0,15603,...,False,False,False,False,False,False,False,False,False,False
1,40000,60,16.14,975.71,3,0.0,45000.0,50.53,18.0,34971,...,False,False,False,False,False,False,False,False,False,False
2,20000,36,7.56,622.68,1,10.0,100000.0,18.92,9.0,25416,...,False,False,False,False,False,False,False,False,False,False
3,4500,36,11.31,147.99,2,10.0,38500.0,4.64,12.0,4472,...,False,False,False,False,False,False,False,False,False,False
4,8425,36,27.27,345.18,5,3.0,450000.0,12.37,21.0,36812,...,False,False,False,False,False,False,False,False,False,False


In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

X_train.shape, X_test.shape

((1042885, 30), (260722, 30))

In [15]:
pip install scikit-learn

  Using cached scikit_learn-1.9.0-cp313-cp313-win_amd64.whl.metadata (11 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
Using cached scikit_learn-1.9.0-cp313-cp313-win_amd64.whl (8.2 MB)
Using cached cloudpickle-3.1.2-py3-none-any.whl (22 kB)
   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   - -------------------------------------- 1.0/36.6 MB 5.4 MB/s eta 0:00:07
   -- ------------------------------------- 2.1/36.6 MB 5.5 MB/s eta 0:00:07
   --- ------------------------------------ 2.9/36.6 MB 5.6 MB/s eta 0:00:07
   --- ------------------------------------ 2.9/36.6 MB 5.6 MB/s eta 0:00:07
   --- ------------------------------------ 2.9/36.6 MB 5.6 MB/s eta 0:00:07
   --- ------------------------------------ 2.9/36.6 MB 5.6 MB/s eta 0:00:07
   --- ------------------------------------ 2.9/36.6 MB 5.6 MB/s eta 0:00:07
   --- ------------------------------------ 2.9/36.6 M


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
